# Code Collision Check

Finds every NCIt C-code that is used in more than one **role**, and every code
that is used more than once **within** a role.

Why it matters: in a merged graph (`cosmos-rdf` + `usdm-rdf` + CT), a
transitive predicate such as `skos:exactMatch` on a shared C-code makes the two
subjects equivalent. Nothing in the sources asserts that. This is the hub
inference problem: two terms that both map to one concept become equivalent to
each other.

Roles collected here:

| Role | Subject | Source |
|---|---|---|
| `bc_identity` | BC | COSMoS_Graph / BC |
| `dec_identity` | Data Element Concept | COSMoS_Graph / DataElementConcepts |
| `testcd_identity` | TESTCD | SDTM_Test_Identity / Test Codes |
| `codelist_identity` | codelist | COSMoS_Graph_CT / Codelists |
| `codelist_term` | codelist membership | COSMoS_Graph_CT / CodelistTerms |
| `pinned_value` | DSS variable | COSMoS_Graph / Variables |
| `instrument_codelist` | instrument codelist | SDTM_Instrument_Identity |
| `instrument_anchor` | NCIt instrument concept (C20993) | SDTM_Instrument_Identity |
| `container_anchor` | NCIt question container (C211913) | SDTM_Instrument_Identity |
| `usdm_anchor` | USDM class or attribute | usdm-rdf / usdm_v4.ttl |

All counts are measured. Nothing is asserted that is not in the files.

In [ ]:
from pathlib import Path

import pandas as pd
from rdflib import Graph, Namespace
from rdflib.namespace import SKOS

REPO = Path.cwd().parent.parent  # notebooks/ -> link-semantics/ -> repo root
USDM_TTL = REPO.parent / "usdm-rdf" / "usdm_v4.ttl"
OUT = REPO / "link-semantics" / "interim" / "Code_Collision_Check.xlsx"

# Optional: path to a previous Code_Collision_Check.xlsx for the version-bump diff in
# the final cell. Leave as None to skip.
PRIOR_FILE = None

GRAPH = REPO / "cosmos-graph" / "interim" / "COSMoS_Graph.xlsx"
GRAPH_CT = REPO / "cosmos-graph" / "interim" / "COSMoS_Graph_CT.xlsx"
TEST_IDENTITY = REPO / "sdtm-test-codes" / "machine_actionable" / "SDTM_Test_Identity.xlsx"
INSTRUMENT_IDENTITY = REPO / "sdtm-test-codes" / "machine_actionable" / "SDTM_Instrument_Identity.xlsx"

for path in [GRAPH, GRAPH_CT, TEST_IDENTITY, INSTRUMENT_IDENTITY, USDM_TTL]:
    if not path.exists():
        raise FileNotFoundError(f"{path} not found - adjust the paths above")

print("repo:", REPO)
print("usdm:", USDM_TTL)

## Collect roles

One row per (code, role, subject). A code used by several subjects in the same
role, or in several roles, shows up as several rows.

In [ ]:
def roles_from_sheet(path, sheet_name, role, subject_col, code_col):
    df = pd.read_excel(path, sheet_name=sheet_name)
    df = df[[subject_col, code_col]].dropna()
    df.columns = ["subject", "ncit_code"]
    df["role"] = role
    df["source"] = f"{path.name}:{sheet_name}"
    return df[["ncit_code", "role", "subject", "source"]]


frames = [
    roles_from_sheet(GRAPH, "BC", "bc_identity", "bc_id", "ncit_code"),
    roles_from_sheet(GRAPH, "DataElementConcepts", "dec_identity", "dec_id", "ncit_dec_code"),
    roles_from_sheet(TEST_IDENTITY, "Test Codes", "testcd_identity", "TESTCD", "NCIt_Code"),
    roles_from_sheet(GRAPH_CT, "Codelists", "codelist_identity", "codelist_submission_value", "codelist_concept_id"),
    roles_from_sheet(GRAPH_CT, "CodelistTerms", "codelist_term", "codelist_submission_value", "term_concept_id"),
    roles_from_sheet(INSTRUMENT_IDENTITY, "Instruments", "instrument_codelist", "Codelist_TESTCD", "Codelist_NCIt_Code"),
    roles_from_sheet(INSTRUMENT_IDENTITY, "Instruments", "instrument_anchor", "Codelist_TESTCD", "Instrument_NCIt_Code"),
    roles_from_sheet(INSTRUMENT_IDENTITY, "Instruments", "container_anchor", "Codelist_TESTCD", "Container_NCIt_Code"),
]

variables = pd.read_excel(GRAPH, sheet_name="Variables")
pinned = variables[["ds_id", "variable_name", "assigned_term_concept_id"]].dropna()
pinned["subject"] = pinned["ds_id"] + "." + pinned["variable_name"]
pinned = pinned.rename(columns={"assigned_term_concept_id": "ncit_code"})
pinned["role"] = "pinned_value"
pinned["source"] = "COSMoS_Graph.xlsx:Variables"
frames.append(pinned[["ncit_code", "role", "subject", "source"]])

roles = pd.concat(frames, ignore_index=True)
roles["ncit_code"] = roles["ncit_code"].astype(str).str.strip()
print(len(roles), "role rows before USDM")

## Add the USDM anchors

`usdm_v4.ttl` anchors classes and attributes to NCIt with `skos:exactMatch`,
in two namespaces (`ncit:` and `obo:NCIT_`). Both are collapsed to the bare
C-code here.

In [ ]:
NCIT = Namespace("http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#")
OBO = Namespace("http://purl.obolibrary.org/obo/")

usdm = Graph()
usdm.parse(USDM_TTL, format="turtle")

usdm_rows = []
for subject, obj in usdm.subject_objects(SKOS.exactMatch):
    value = str(obj)
    if value.startswith(str(NCIT)):
        code_value = value[len(str(NCIT)):]
    elif value.startswith(str(OBO) + "NCIT_"):
        code_value = value[len(str(OBO) + "NCIT_"):]
    else:
        continue
    usdm_rows.append({
        "ncit_code": code_value,
        "role": "usdm_anchor",
        "subject": str(subject).rsplit("/", 1)[-1],
        "source": USDM_TTL.name,
    })

usdm_roles = pd.DataFrame(usdm_rows).drop_duplicates()
roles = pd.concat([roles, usdm_roles], ignore_index=True)

print(len(usdm_roles), "USDM anchor rows,", usdm_roles["ncit_code"].nunique(), "distinct codes")
print(len(roles), "role rows total,", roles["ncit_code"].nunique(), "distinct codes")

## Pins

What this run was executed against. The SDTM CT release date is not recorded
in any of the input files, so it cannot be pinned from data here.

In [ ]:
import subprocess

from rdflib.namespace import OWL, RDF


def commit_of(path):
    return subprocess.run(
        ["git", "rev-parse", "HEAD"], cwd=path, capture_output=True, text=True, check=True
    ).stdout.strip()


package_dates = pd.read_excel(GRAPH, sheet_name="DSS")["package_date"]
usdm_versions = sorted(str(value) for value in usdm.objects(None, OWL.versionInfo))

pins = pd.DataFrame(
    [
        {"pin": "cdisc-for-ai commit", "value": commit_of(REPO)},
        {"pin": "usdm-rdf commit", "value": commit_of(USDM_TTL.parent)},
        {"pin": "usdm_v4.ttl owl:versionInfo", "value": " | ".join(usdm_versions)},
        {"pin": "usdm_v4.ttl triples", "value": str(len(usdm))},
        {"pin": "COSMoS package_date (latest in DSS sheet)", "value": str(package_dates.max())},
    ]
)
pins

### What the USDM anchors sit on

The OWL type of each anchored subject in `usdm_v4.ttl`. SKOS defines its
mapping properties between concepts; the count shows how many anchors sit on a
class and how many on a property.

In [ ]:
usdm_kind = {}
for subject in set(usdm.subjects(SKOS.exactMatch)):
    kinds = sorted(
        str(kind)[len(str(OWL)):] for kind in usdm.objects(subject, RDF.type) if str(kind).startswith(str(OWL))
    )
    if len(kinds) != 1:
        raise ValueError(f"{subject} has {len(kinds)} OWL types: {kinds}")
    usdm_kind[str(subject).rsplit("/", 1)[-1]] = kinds[0]

usdm_roles["usdm_kind"] = usdm_roles["subject"].map(usdm_kind)
usdm_anchor_types = (
    usdm_roles.groupby("usdm_kind", as_index=False).agg(anchors=("ncit_code", "nunique"))
)
usdm_anchor_types

## A. Within-role duplicates

Same code, same role, different subjects. Under a transitive predicate these
subjects become equivalent to each other.

In [ ]:
within = (
    roles.drop_duplicates(["ncit_code", "role", "subject"])
    .groupby(["role", "ncit_code"], as_index=False)
    .agg(subjects=("subject", "nunique"), examples=("subject", lambda s: " | ".join(sorted(s)[:4])))
)
within_dups = within[within["subjects"] > 1].sort_values(
    ["role", "subjects", "ncit_code"], ascending=[True, False, True]
)

within_summary = (
    within_dups.groupby("role", as_index=False)
    .agg(codes_with_duplicates=("ncit_code", "nunique"), max_subjects=("subjects", "max"))
)
within_summary

In [ ]:
within_dups.head(25)

## A2. Subjects with several codes

The inverse of A: one subject string carrying more than one C-code within an
identity role. The code stays a clean identifier, but a join on the subject
string is ambiguous.

In [ ]:
identity_roles = ["bc_identity", "dec_identity", "testcd_identity", "codelist_identity"]
codes_per_subject = (
    roles[roles["role"].isin(identity_roles)]
    .groupby(["role", "subject"], as_index=False)
    .agg(codes=("ncit_code", "nunique"))
)
several_codes_summary = (
    codes_per_subject.groupby("role", as_index=False)
    .agg(subjects=("subject", "size"), subjects_with_several_codes=("codes", lambda s: int((s > 1).sum())))
)
several_codes_summary

In [ ]:
test_codes = pd.read_excel(TEST_IDENTITY, sheet_name="Test Codes")
ambiguous_testcd = codes_per_subject[
    (codes_per_subject["role"] == "testcd_identity") & (codes_per_subject["codes"] > 1)
]["subject"]
testcd_several_codes = (
    test_codes[test_codes["TESTCD"].isin(ambiguous_testcd)][["TESTCD", "NCIt_Code", "TEST", "SDTM_Domains"]]
    .sort_values(["TESTCD", "NCIt_Code"])
    .reset_index(drop=True)
)
print(testcd_several_codes["TESTCD"].nunique(), "TESTCD strings,", len(testcd_several_codes), "rows")
testcd_several_codes

## B. Cross-role usage

How many distinct codes each pair of roles shares. `codelist_term` is expected
to overlap with everything, since membership is how a concept enters a list.
The interesting pairs are the ones between *identity* roles.

In [ ]:
role_sets = roles.groupby("role")["ncit_code"].apply(set).to_dict()
role_names = sorted(role_sets)

pairs = []
for i, left in enumerate(role_names):
    for right in role_names[i + 1:]:
        shared = role_sets[left] & role_sets[right]
        if shared:
            pairs.append({
                "role_a": left,
                "role_b": right,
                "shared_codes": len(shared),
                "examples": " | ".join(sorted(shared)[:5]),
            })

role_pairs = pd.DataFrame(pairs).sort_values(
    ["shared_codes", "role_a", "role_b"], ascending=[False, True, True]
)
role_pairs

In [ ]:
role_counts = (
    roles.drop_duplicates(["ncit_code", "role"])
    .groupby("ncit_code", as_index=False)
    .agg(roles_used=("role", "nunique"), role_list=("role", lambda s: " | ".join(sorted(s))))
)
multi_role = role_counts[role_counts["roles_used"] > 1]

print(len(role_counts), "distinct codes")
print(len(multi_role), "codes used in more than one role")
multi_role["role_list"].value_counts().head(15)

### BC and DEC on the same code

Detail behind the `bc_identity` x `dec_identity` pair. `same_bc` marks the rows
where the DEC belongs to the BC that carries the same code.

In [ ]:
bc_sheet = pd.read_excel(GRAPH, sheet_name="BC")
dec_sheet = pd.read_excel(GRAPH, sheet_name="DataElementConcepts")

bc_dec = (
    bc_sheet[["bc_id", "ncit_code", "bc_short_name"]].dropna(subset=["ncit_code"])
    .merge(
        dec_sheet[["bc_id", "dec_id", "ncit_dec_code", "dec_label"]].dropna(subset=["ncit_dec_code"]),
        left_on="ncit_code",
        right_on="ncit_dec_code",
        suffixes=("", "_of_dec"),
    )
)
bc_dec["same_bc"] = bc_dec["bc_id"] == bc_dec["bc_id_of_dec"]
bc_dec["same_label"] = bc_dec["bc_short_name"] == bc_dec["dec_label"]
bc_dec = bc_dec.sort_values(["ncit_code", "bc_id_of_dec"]).reset_index(drop=True)

bc_dec_summary = pd.DataFrame(
    [
        {"measure": "BCs without a code (dropped before the join)", "count": int(bc_sheet["ncit_code"].isna().sum())},
        {"measure": "DEC rows without a code (dropped before the join)", "count": int(dec_sheet["ncit_dec_code"].isna().sum())},
        {"measure": "shared codes", "count": bc_dec["ncit_code"].nunique()},
        {"measure": "BC x DEC rows on a shared code", "count": len(bc_dec)},
        {"measure": "... rows where the DEC belongs to the same BC", "count": int(bc_dec["same_bc"].sum())},
        {"measure": "... codes where the DEC belongs to the same BC", "count": bc_dec[bc_dec["same_bc"]]["ncit_code"].nunique()},
        {"measure": "... rows with identical labels", "count": int(bc_dec["same_label"].sum())},
    ]
)
bc_dec_summary

In [ ]:
bc_dec[bc_dec["same_bc"]][["ncit_code", "bc_short_name", "dec_label", "same_label"]]

### BCs on a question-container code

Detail behind the `bc_identity` x `container_anchor` pair: what these BCs look
like in the COSMoS content, and whether the instrument concept of the same
instrument codelist is their parent.

In [ ]:
instruments = pd.read_excel(INSTRUMENT_IDENTITY, sheet_name="Instruments")
bc_parents = pd.read_excel(GRAPH, sheet_name="BC_Parents")
variables_sheet = pd.read_excel(GRAPH, sheet_name="Variables")
code_of_bc = bc_sheet.set_index("bc_id")["ncit_code"]

container_bcs = (
    bc_sheet[["bc_id", "ncit_code", "bc_short_name", "bc_type", "bc_categories"]].dropna(subset=["ncit_code"])
    .merge(
        instruments[["Codelist_TESTCD", "Container_NCIt_Code", "Instrument_NCIt_Code", "Instrument_Match_Method"]]
        .dropna(subset=["Container_NCIt_Code"]),
        left_on="ncit_code",
        right_on="Container_NCIt_Code",
    )
    .merge(bc_parents[["bc_id", "parent_bc_id", "parent_bc_short_name"]], on="bc_id", how="left")
)
if container_bcs["bc_id"].duplicated().any():
    raise ValueError("a container BC appears more than once - several parents or several instrument codelists")
container_bcs["children"] = container_bcs["bc_id"].map(bc_parents["parent_bc_id"].value_counts()).fillna(0).astype(int)
container_bcs["dss_variable_rows"] = container_bcs["bc_id"].map(variables_sheet["bc_id"].value_counts()).fillna(0).astype(int)
container_bcs["dec_rows"] = container_bcs["bc_id"].map(dec_sheet["bc_id"].value_counts()).fillna(0).astype(int)
container_bcs["instrument_is_bc"] = container_bcs["Instrument_NCIt_Code"].isin(bc_sheet["ncit_code"].dropna())
container_bcs["parent_is_instrument"] = (
    container_bcs["parent_bc_id"].map(code_of_bc) == container_bcs["Instrument_NCIt_Code"]
)

container_summary = pd.DataFrame(
    [
        {"measure": "BCs on a question-container code", "count": container_bcs["bc_id"].nunique()},
        {"measure": "... bc_type values", "count": " | ".join(f"{k}: {v}" for k, v in container_bcs["bc_type"].value_counts().items())},
        {"measure": "... bc_categories values", "count": " | ".join(f"{k}: {v}" for k, v in container_bcs["bc_categories"].value_counts().items())},
        {"measure": "... parents", "count": " | ".join(f"{k}: {v}" for k, v in container_bcs["parent_bc_id"].value_counts().items())},
        {"measure": "... DSS variable rows", "count": int(container_bcs["dss_variable_rows"].sum())},
        {"measure": "... DEC rows", "count": int(container_bcs["dec_rows"].sum())},
        {"measure": "... child BCs", "count": int(container_bcs["children"].sum())},
        {"measure": "... with an instrument anchor (C20993)", "count": int(container_bcs["Instrument_NCIt_Code"].notna().sum())},
        {"measure": "... without an instrument anchor", "count": int(container_bcs["Instrument_NCIt_Code"].isna().sum())},
        {"measure": "... whose instrument concept is also a BC", "count": int(container_bcs["instrument_is_bc"].sum())},
        {"measure": "... whose parent is the instrument concept", "count": int(container_bcs["parent_is_instrument"].sum())},
    ]
)
container_summary

## C. USDM x COSMoS overlap

Codes that anchor a USDM class or attribute *and* appear somewhere in the
COSMoS or CT content. These are the ones that merge the two repos.

In [ ]:
usdm_codes = role_sets.get("usdm_anchor", set())
cosmos_roles = [name for name in role_names if name != "usdm_anchor"]

overlap_rows = []
for name in cosmos_roles:
    shared = usdm_codes & role_sets[name]
    if shared:
        overlap_rows.append({
            "cosmos_role": name,
            "shared_codes": len(shared),
            "examples": " | ".join(sorted(shared)[:8]),
        })

usdm_overlap = pd.DataFrame(overlap_rows).sort_values(
    ["shared_codes", "cosmos_role"], ascending=[False, True]
)
usdm_overlap

In [ ]:
shared_all = sorted(usdm_codes & set().union(*[role_sets[name] for name in cosmos_roles]))
detail = roles[roles["ncit_code"].isin(shared_all)].drop_duplicates(["ncit_code", "role", "subject"])
detail = detail.sort_values(["ncit_code", "role"])
print(len(shared_all), "codes shared between usdm_anchor and COSMoS/CT roles")
detail.head(40)

In [ ]:
detail.merge(usdm_roles[["ncit_code", "usdm_kind"]], on="ncit_code").drop_duplicates("ncit_code")["usdm_kind"].value_counts()

### USDM-anchored codes that COSMoS pins as a value

Detail behind the `pinned_value` x `usdm_anchor` pair: which DSS variable pins
the code, and what kind of USDM element carries the same code.

In [ ]:
pinned_usdm = sorted(role_sets["pinned_value"] & usdm_codes)
pinned_rows = roles[(roles["role"] == "pinned_value") & roles["ncit_code"].isin(pinned_usdm)].copy()
pinned_rows["ds_id"] = pinned_rows["subject"].str.rsplit(".", n=1).str[0]
pinned_rows["variable_name"] = pinned_rows["subject"].str.rsplit(".", n=1).str[1]

ts_bridge = (
    pinned_rows.groupby("ncit_code", as_index=False)
    .agg(
        ds_ids=("ds_id", lambda s: " | ".join(sorted(set(s)))),
        variables=("variable_name", lambda s: " | ".join(sorted(set(s)))),
        pins=("subject", "size"),
    )
    .merge(usdm_roles[["ncit_code", "subject", "usdm_kind"]].rename(columns={"subject": "usdm_subject"}), on="ncit_code")
)
print(len(ts_bridge), "codes;", "variables pinned:", sorted(set(pinned_rows["variable_name"])))
print(ts_bridge["usdm_kind"].value_counts().to_dict())
ts_bridge

## Key measures

Long format (measure, value), the diffable part of this notebook. A package
that introduces a duplicate identity code, or moves any of these, shows up in
the final cell as a diff - a finding, not a failure.

In [ ]:
def measures(df, prefix, label_cols, value_col):
    labels = df[label_cols].astype(str).agg(" / ".join, axis=1)
    return pd.DataFrame({"measure": prefix + ": " + labels, "value": df[value_col].astype(int)})


role_sizes = roles.drop_duplicates(["ncit_code", "role"]).groupby("role", as_index=False).agg(codes=("ncit_code", "size"))
multi_role_total = pd.DataFrame([{"measure": "codes used in more than one role", "value": len(multi_role)}])
identity_pairs = role_pairs[role_pairs["role_a"].isin(identity_roles) & role_pairs["role_b"].isin(identity_roles)]
container_counts = container_summary[container_summary["count"].map(lambda v: isinstance(v, int))]
usdm_shared_total = pd.DataFrame([{"measure": "codes shared between usdm_anchor and COSMoS/CT roles", "value": len(shared_all)}])
pinned_usdm_total = pd.DataFrame([{"measure": "USDM-anchored codes pinned as a value", "value": len(ts_bridge)}])

key_measures = pd.concat(
    [
        measures(role_sizes, "distinct codes per role", ["role"], "codes"),
        measures(within_summary, "within-role duplicates: codes", ["role"], "codes_with_duplicates"),
        measures(within_summary, "within-role duplicates: max subjects", ["role"], "max_subjects"),
        measures(several_codes_summary, "subjects with several codes", ["role"], "subjects_with_several_codes"),
        multi_role_total,
        measures(identity_pairs, "shared codes between identity roles", ["role_a", "role_b"], "shared_codes"),
        measures(bc_dec_summary, "BC x DEC", ["measure"], "count"),
        measures(container_counts, "container BCs", ["measure"], "count"),
        measures(usdm_anchor_types, "USDM anchors", ["usdm_kind"], "anchors"),
        measures(usdm_overlap, "USDM x COSMoS shared codes", ["cosmos_role"], "shared_codes"),
        usdm_shared_total,
        pinned_usdm_total,
    ],
    ignore_index=True,
)
if key_measures["measure"].duplicated().any():
    raise ValueError("duplicate measure names: " + ", ".join(key_measures.loc[key_measures["measure"].duplicated(), "measure"]))
print(len(key_measures), "key measures")
key_measures

## Write the workbook

In [ ]:
with pd.ExcelWriter(OUT, engine="openpyxl") as writer:
    within_summary.to_excel(writer, sheet_name="Within_Role_Summary", index=False)
    within_dups.to_excel(writer, sheet_name="Within_Role_Duplicates", index=False)
    role_pairs.to_excel(writer, sheet_name="Role_Pairs", index=False)
    multi_role.to_excel(writer, sheet_name="Multi_Role_Codes", index=False)
    usdm_overlap.to_excel(writer, sheet_name="USDM_Overlap", index=False)
    detail.to_excel(writer, sheet_name="USDM_Overlap_Detail", index=False)
    roles.to_excel(writer, sheet_name="Role_Inventory", index=False)
    pins.to_excel(writer, sheet_name="Pins", index=False)
    usdm_anchor_types.to_excel(writer, sheet_name="USDM_Anchor_Types", index=False)
    several_codes_summary.to_excel(writer, sheet_name="Several_Codes_Summary", index=False)
    testcd_several_codes.to_excel(writer, sheet_name="TESTCD_Several_Codes", index=False)
    bc_dec_summary.to_excel(writer, sheet_name="BC_DEC_Summary", index=False)
    bc_dec.to_excel(writer, sheet_name="BC_DEC_Shared_Code", index=False)
    container_summary.to_excel(writer, sheet_name="Container_BC_Summary", index=False)
    container_bcs.to_excel(writer, sheet_name="Container_BCs", index=False)
    ts_bridge.to_excel(writer, sheet_name="USDM_Pinned_Values", index=False)
    key_measures.to_excel(writer, sheet_name="Key_Measures", index=False)

print("written:", OUT)

## How to read this

- **Within-role duplicates** are the direct hub risk. Two BCs on one C-code, or
  two TESTCDs on one C-code, become equivalent under a transitive predicate.
- **Cross-role codes** are the subtler case: the same concept plays identity in
  one place and a pinned value or a codelist term in another. A single
  predicate for all of them merges those roles.
- **Subjects with several codes** is the inverse case: the code is unique, but
  the string a consumer joins on is not. Only TESTCD shows it.
- **USDM overlap** is the cross-repo case. Any code in this list is a join
  point between `usdm-rdf` and `cosmos-rdf` in a merged graph.

None of this is an error in the sources. It only becomes one if the RDF
rendering uses a transitive predicate where the source means identity,
membership, denotation or a pinned value.

## Version-bump diff (optional)

Set `PRIOR_FILE` (cell 2) to a previous `Code_Collision_Check.xlsx` to compare
its `Key_Measures` against this run. No-ops if unset or missing.

In [ ]:
if PRIOR_FILE and Path(PRIOR_FILE).exists():
    prior = pd.read_excel(PRIOR_FILE, sheet_name="Key_Measures")
    prev = dict(zip(prior["measure"], prior["value"]))
    curr = dict(zip(key_measures["measure"], key_measures["value"]))
    added = sorted(set(curr) - set(prev))
    dropped = sorted(set(prev) - set(curr))
    changed = [(m, prev[m], curr[m]) for m in sorted(set(prev) & set(curr)) if prev[m] != curr[m]]
    print(f"Prior:    {PRIOR_FILE}")
    print(f"Measures: prior {len(prev)} -> current {len(curr)}")
    print(f"Added measures:   {added}")
    print(f"Dropped measures: {dropped}")
    print("Value changes (measure: prior -> current):")
    for m, a, b in changed:
        print(f"  {m}: {a} -> {b}")
    if not changed:
        print("  (none)")
else:
    print("PRIOR_FILE not set or missing - skipping version diff.")